# Transformación de datasets para la capa Plata

## Importación de librerías

In [ ]:
import os
import io
import json
import glob
import boto3
import pandas as pd
from dotenv import load_dotenv

load_dotenv(override=True)

ACCESS_KEY = os.getenv("AWS_ACCESS_KEY_ID")
SECRET_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
SESSION_TOKEN = os.getenv("AWS_SESSION_TOKEN")

## Datasets de Demanda Energética (ESIOS)

### Carga de ficheros y unificación en un mismo dataframe

In [ ]:
BUCKET_NAME = "predictor-demanda-electrica-marcff-2026"
PREFIX_DEMANDA = "bronze/esios/"

# Creamos el cliente de S3
s3_client = boto3.client("s3")

print(f"Buscando archivos de demanda en el bucket '{BUCKET_NAME}'...")

# Listamos los objetos dentro de la carpeta de demanda
paginator = s3_client.get_paginator("list_objects_v2")
paginas = paginator.paginate(Bucket=BUCKET_NAME, Prefix=PREFIX_DEMANDA)

lista_df = []

for pagina in paginas:
    if "Contents" in pagina:
        for objeto in pagina["Contents"]:
            ruta_s3 = objeto["Key"]
            
            # Aseguramos que solo lea los archivos JSON
            if ruta_s3.endswith(".json"):
                # Descargamos el archivo directamente a la memoria de Python
                respuesta = s3_client.get_object(Bucket=BUCKET_NAME, Key=ruta_s3)
                contenido_json = json.loads(respuesta["Body"].read().decode("utf-8"))
                
                # Accedemos a la estructura de ESIOS
                if 'indicator' in contenido_json and 'values' in contenido_json['indicator']:
                    valores = contenido_json['indicator']['values']
                    df_mes = pd.DataFrame(valores)
                    lista_df.append(df_mes)

# Concatenamos todos los fragmentos en un solo Dataframe
if lista_df:
    df_demanda = pd.concat(lista_df, ignore_index=True)
    print(f"Dataset de Demanda energética cargado desde S3. Total de filas: {df_demanda.shape[0]}")
    display(df_demanda.head(100))
else:
    print("No se encontraron archivos JSON en la ruta especificada de S3.")

Buscando archivos de demanda en el bucket 'predictor-demanda-electrica-marcff-2026'...
Dataset de Demanda energética cargado desde S3. Total de filas: 820968


,value,datetime,datetime_utc,tz_time,geo_id,geo_name
0,24546.0,2014-01-01T00:00:00.000+01:00,2013-12-31T23:00:00Z,2013-12-31T23:00:00.000Z,8741,Península
1,24309.0,2014-01-01T00:10:00.000+01:00,2013-12-31T23:10:00Z,2013-12-31T23:10:00.000Z,8741,Península
2,24348.0,2014-01-01T00:20:00.000+01:00,2013-12-31T23:20:00Z,2013-12-31T23:20:00.000Z,8741,Península
3,24321.0,2014-01-01T00:30:00.000+01:00,2013-12-31T23:30:00Z,2013-12-31T23:30:00.000Z,8741,Península
4,24194.0,2014-01-01T00:40:00.000+01:00,2013-12-31T23:40:00Z,2013-12-31T23:40:00.000Z,8741,Península
...,...,...,...,...,...,...
95,23084.0,2014-01-01T15:50:00.000+01:00,2014-01-01T14:50:00Z,2014-01-01T14:50:00.000Z,8741,Península
96,22985.0,2014-01-01T16:00:00.000+01:00,2014-01-01T15:00:00Z,2014-01-01T15:00:00.000Z,8741,Península
97,22787.0,2014-01-01T16:10:00.000+01:00,2014-01-01T15:10:00Z,2014-01-01T15:10:00.000Z,8741,Península
98,22718.0,2014-01-01T16:20:00.000+01:00,2014-01-01T15:20:00Z,2014-01-01T15:20:00.000Z,8741,Península


### Análisis exploratorio (EDA) del dataframe resultante

In [3]:
# Tipos de datos y memoria
print("\nEstructura y Tipos de Datos:")
df_demanda.info()

# Recuento de Nulos
print("\n\nValores Nulos por Columna:")
nulos_demanda = df_demanda.isnull().sum()
print(nulos_demanda[nulos_demanda > 0])

# Resumen Estadístico
print("\n\nResumen Estadístico (Buscando anomalías como consumos negativos):")
# Convertimos temporalmente el 'value' a numérico solo para el describe, por si viniera como texto
display(pd.to_numeric(df_demanda['value'], errors='coerce').describe())


Estructura y Tipos de Datos:
<class 'pandas.DataFrame'>
RangeIndex: 820968 entries, 0 to 820967
Data columns (total 6 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   value         820968 non-null  float64
 1   datetime      820968 non-null  str    
 2   datetime_utc  820968 non-null  str    
 3   tz_time       820968 non-null  str    
 4   geo_id        820968 non-null  int64  
 5   geo_name      820968 non-null  str    
dtypes: float64(1), int64(1), str(4)
memory usage: 102.6 MB


Valores Nulos por Columna:
Series([], dtype: int64)


Resumen Estadístico (Buscando anomalías como consumos negativos):


count    820968.000000
mean      27476.673831
std        4516.187522
min           0.000000
25%       23797.000000
50%       27439.000000
75%       30880.000000
max       42052.000000
Name: value, dtype: float64

### Creación de una nueva columna Fecha/Hora para el cruce con el resto de dataframes

In [ ]:
df_demanda['datetime'] = pd.to_datetime(df_demanda['datetime'], utc=True)

# Convertimos el huso horario al de la Península
df_demanda['datetime_local'] = df_demanda['datetime'].dt.tz_convert('Europe/Madrid')

# Extraemos el valor limpio de la demanda
df_demanda['demanda_mw'] = df_demanda['value'].astype(float)

# Creamos una columna clave 'fecha_cruce' con formato YYYY-MM-DD
df_demanda['fecha_cruce'] = df_demanda['datetime_local'].dt.strftime('%Y-%m-%d')

#Limpiamos datetime_local para quedarnos solo con la hora en formato HH:MM:SS
df_demanda['hora'] = df_demanda['datetime_local'].dt.strftime('%H:%M:%S')

# Limpiamos el DataFrame para quedarnos solo con lo que va a la Capa Plata
df_demanda_plata = df_demanda[['fecha_cruce', 'hora', 'demanda_mw']].copy()

print("Estandarización horaria completada. Filas listas:", df_demanda_plata.shape[0])
display(df_demanda_plata.head(100))

Estandarización horaria completada. Filas listas: 820968


,fecha_cruce,hora,demanda_mw
0,2014-01-01,00:00:00,24546.0
1,2014-01-01,00:10:00,24309.0
2,2014-01-01,00:20:00,24348.0
3,2014-01-01,00:30:00,24321.0
4,2014-01-01,00:40:00,24194.0
...,...,...,...
95,2014-01-01,15:50:00,23084.0
96,2014-01-01,16:00:00,22985.0
97,2014-01-01,16:10:00,22787.0
98,2014-01-01,16:20:00,22718.0


## Dataset de Histórico Climático de la AEMET

### Carga de ficheros y unificación en un mismo dataframe


In [5]:
PREFIX_CLIMA = "bronze/clima/" 

paginator = s3_client.get_paginator("list_objects_v2")
paginas = paginator.paginate(Bucket=BUCKET_NAME, Prefix=PREFIX_CLIMA)

lista_clima = []
contador_archivos = 0

for pagina in paginas:
    if "Contents" in pagina:
        for objeto in pagina["Contents"]:
            ruta_s3 = objeto["Key"]
            
            # Filtramos para leer solo los xls de la AEMET
            if ruta_s3.endswith(".xls") or "Aemet" in ruta_s3:
                contador_archivos += 1
                
                # Extraemos la fecha del nombre del archivo
                nombre_fichero = ruta_s3.split('/')[-1]
                fecha_archivo = nombre_fichero.replace(".xls", "").replace(".xlsx", "").replace("Aemet", "")
                
                respuesta = s3_client.get_object(Bucket=BUCKET_NAME, Key=ruta_s3)
                
                # Descargamos los bytes a la memoria
                bytes_file = io.BytesIO(respuesta["Body"].read())
                
                df_dia = None
                
                # Aunque los archivos tengan extensión .xls, algunos en realidad son HTML. Montamos un doble intento de lectura para manejar ambos casos. 
                try:
                    # Primero intentamos leer como Excel real (.xls binario)
                    df_dia = pd.read_excel(bytes_file, engine='xlrd')
                    
                except Exception as e_excel:
                    # Si no es un Excel, intentamos leerlo como HTML
                    bytes_file.seek(0) 
                    
                    try:
                        tablas_html = pd.read_html(bytes_file)
                        if tablas_html:
                            df_dia = tablas_html[0] 
                    except Exception as e_html:
                        print(f"Archivo corrupto ignorado ({ruta_s3}). No es ni Excel ni HTML.")
                
                # Inyectamos la fecha y guardamos si la lectura fue un éxito
                if df_dia is not None and not df_dia.empty:
                    df_dia['fecha_cruce'] = fecha_archivo
                    lista_clima.append(df_dia)

if lista_clima:
    df_clima = pd.concat(lista_clima, ignore_index=True)
    print(f"Dataset cargado. Procesados {contador_archivos} archivos diarios.")
    print(f"Total de filas cargadas: {df_clima.shape[0]}")
    display(df_clima.head())
else:
    print("No se ha podido extraer información del climática.")

Archivo corrupto ignorado (bronze/clima/year=2014/month=04/day=07/Aemet2014-04-07.xls). No es ni Excel ni HTML.
Archivo corrupto ignorado (bronze/clima/year=2020/month=07/day=26/Aemet2020-07-26.xls). No es ni Excel ni HTML.
Archivo corrupto ignorado (bronze/clima/year=2022/month=09/day=27/Aemet2022-09-27.xls). No es ni Excel ni HTML.
Archivo corrupto ignorado (bronze/clima/year=2022/month=09/day=28/Aemet2022-09-28.xls). No es ni Excel ni HTML.
Dataset cargado. Procesados 4376 archivos diarios.
Total de filas cargadas: 3517428


,España,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,fecha_cruce
0,"Actualizado: jueves, 02 enero 2014 a las 09:00...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2014-01-01
1,"Fecha: miércoles, 01 enero 2014",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2014-01-01
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2014-01-01
3,Estación,Provincia,Temperatura máxima (ºC),Temperatura mínima (ºC),Temperatura media (ºC),Racha (km/h),Velocidad máxima (km/h),Precipitación 00-24h (mm),Precipitación 00-06h (mm),Precipitación 06-12h (mm),Precipitación 12-18h (mm),Precipitación 18-24h (mm),2014-01-01
4,Estaca de Bares,A Coruña,14.5 (12:20),10.0 (00:00),12.3,133 (14:40),92 (13:00),3,0,0.4,2.4,0.2,2014-01-01


Como podemos observar, la cabecera del dataset es errónea, así que antes de realizar el análisis estadístico, procedemos a solucionar esto con el fin de poder seguir limpiando los datos

In [6]:
df_clima_plata = df_clima.copy()

# Rescatamos los nombres reales de las columnas, ubicados en la fila 4 (índice 3). fecha_cruce está bien formateada porque la creamos nosotros, así que lo corregimos a mano.
columnas_reales = df_clima_plata.iloc[3].tolist()
columnas_reales[-1] = 'fecha_cruce'
print(f"Columnas reales: {columnas_reales[:5]}...")

# Machacamos los nombres de las columnas erróneos con los nombres de las columnas reales
df_clima_plata.columns = columnas_reales

# Eliminamos todas las filas que en la columna 'Provincia' tengan un NaN para eliminar las filas informativas ("Actualizado: jueves,02...")
df_clima_plata = df_clima_plata.dropna(subset=['Provincia'])

# Como cada fichero venía con sus cabeceras, eliminamos las que quedaron como filas dentro del dataframe
df_clima_plata = df_clima_plata[df_clima_plata['Provincia'] != 'Provincia']

# Reseteamos los índices para que vuelva a contar desde 0 de forma limpia
df_clima_plata = df_clima_plata.reset_index(drop=True)

display(df_clima_plata.head())

Columnas reales: ['Estación', 'Provincia', 'Temperatura máxima (ºC)', 'Temperatura mínima (ºC)', 'Temperatura media (ºC)']...


,Estación,Provincia,Temperatura máxima (ºC),Temperatura mínima (ºC),Temperatura media (ºC),Racha (km/h),Velocidad máxima (km/h),Precipitación 00-24h (mm),Precipitación 00-06h (mm),Precipitación 06-12h (mm),Precipitación 12-18h (mm),Precipitación 18-24h (mm),fecha_cruce
0,Estaca de Bares,A Coruña,14.5 (12:20),10.0 (00:00),12.3,133 (14:40),92 (13:00),3,0,0.4,2.4,0.2,2014-01-01
1,As Pontes,A Coruña,11.9 (18:30),7.8 (00:10),9.8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2014-01-01
2,A Coruña,A Coruña,14.7 (11:50),10.8 (00:00),12.8,83 (14:20),40 (12:00),8.4,0,2.2,6.2,0,2014-01-01
3,A Coruña Aeropuerto,A Coruña,14.7 (18:10),9.6 (00:50),12.2,98 (14:20),50 (14:20),4.9,0,1.3,2.8,0.8,2014-01-01
4,"Carballo, Depuradora",A Coruña,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2014-01-01


### Análisis estadístico (EDA) del dataset de histórico climático

In [7]:
# Obtenemos información acerca de la estructura y tipos de datos
print("\nEstructura y Tipos de Datos:")
df_clima_plata.info()

# Calculamos los valores nulos por columna
print("\nValores Nulos por Columna:")
nulos_clima = df_clima_plata.isnull().sum()
porcentaje_nulos = (nulos_clima / len(df_clima_plata)) * 100
df_nulos = pd.DataFrame({'Nulos': nulos_clima, 'Porcentaje (%)': porcentaje_nulos.round(2)})
display(df_nulos[df_nulos['Nulos'] > 0].sort_values(by='Porcentaje (%)', ascending=False))

# Comprobamos si hay errores tipográficos en la columna 'Provincia' 
print("\nValores Únicos en 'Provincia'")
if 'Provincia' in df_clima_plata.columns:
    print(df_clima_plata['Provincia'].dropna().unique()[:10])

print(f"Filas resultantes (1 por provincia y día): {df_clima_plata.shape[0]}")


Estructura y Tipos de Datos:
<class 'pandas.DataFrame'>
RangeIndex: 3499940 entries, 0 to 3499939
Data columns (total 13 columns):
 #   Column                     Dtype 
---  ------                     ----- 
 0   Estación                   str   
 1   Provincia                  str   
 2   Temperatura máxima (ºC)    str   
 3   Temperatura mínima (ºC)    str   
 4   Temperatura media (ºC)     object
 5   Racha (km/h)               str   
 6   Velocidad máxima (km/h)    str   
 7   Precipitación 00-24h (mm)  object
 8   Precipitación 00-06h (mm)  object
 9   Precipitación 06-12h (mm)  object
 10  Precipitación 12-18h (mm)  object
 11  Precipitación 18-24h (mm)  object
 12  fecha_cruce                str   
dtypes: object(6), str(7)
memory usage: 585.0+ MB

Valores Nulos por Columna:


,Nulos,Porcentaje (%)
Racha (km/h),702082,20.06
Velocidad máxima (km/h),684622,19.56
Precipitación 00-24h (mm),259435,7.41
Temperatura máxima (ºC),222393,6.35
Temperatura mínima (ºC),222393,6.35
Temperatura media (ºC),222393,6.35
Precipitación 18-24h (mm),198541,5.67
Precipitación 06-12h (mm),190049,5.43
Precipitación 12-18h (mm),188593,5.39
Precipitación 00-06h (mm),177824,5.08



Valores Únicos en 'Provincia'
<ArrowStringArray>
[        'A Coruña',         'Albacete', 'Alacant/Alicante',
          'Almería',      'Araba/Álava',         'Asturias',
            'Ávila',          'Badajoz',        'Barcelona',
          'Bizkaia']
Length: 10, dtype: str
Filas resultantes (1 por provincia y día): 3499940


### Limpieza estructural del dataset de histórico climático

Las columnas referentes al viento tienen demasiados valores nulos. Al tratarse de una variable poco representativa del clima a la hora de calcular el gasto energético.
Por otro lado, hay muchas columnas referentes a las precipitaciones y tampoco son tan relevantes como la temperatura a la hora de calcular el gasto energético, así que prescindiremos de ellas.
Además, eliminamos también la columna de la temperatura media

In [8]:
# Eliminamos los nulos en la columna 'Estación' 
df_clima_plata = df_clima_plata.dropna(subset=['Estación'])

# Limpiamos las columnas temperatrura eliminando información irrelevante en ellas
cols_temperatura = ['Temperatura máxima (ºC)', 'Temperatura mínima (ºC)']
for col in cols_temperatura:
    df_clima_plata[col] = df_clima_plata[col].astype(str).str.split().str[0]    
    df_clima_plata[col] = pd.to_numeric(df_clima_plata[col], errors='coerce')

# Nos quedamos solo con las columnas más relevantes
columnas_relevantes = ['Temperatura máxima (ºC)', 'Temperatura mínima (ºC)','fecha_cruce']
df_clima_plata = df_clima_plata[columnas_relevantes]

# Agrupamos las filas por Fecha, fusionando todas las estaciones en una sola fila por día
df_clima_plata = df_clima_plata.groupby(['fecha_cruce']).mean(numeric_only=True).reset_index()

# Rellenamos los nulos restantes en las columnas de temperatura usando interpolación lineal y cambiamos el formato del nombre de la columna
cols_temperatura = ['Temperatura máxima (ºC)', 'Temperatura mínima (ºC)']
df_clima_plata[cols_temperatura] = df_clima_plata[cols_temperatura].interpolate()
df_clima_plata = df_clima_plata.rename(columns={'Temperatura máxima (ºC)': 'temperatura_maxima', 'Temperatura mínima (ºC)': 'temperatura_minima'})

# Comprobamos el resultado final
print(f"Filas resultantes: {df_clima_plata.shape[0]}")
print("\n¿Cuántos nulos nos han quedado ahora en las temperaturas?")
print(df_clima_plata.isnull().sum())

display(df_clima_plata.head())

Filas resultantes: 4372

¿Cuántos nulos nos han quedado ahora en las temperaturas?
fecha_cruce           0
temperatura_maxima    0
temperatura_minima    0
dtype: int64


,fecha_cruce,temperatura_maxima,temperatura_minima
0,2014-01-01,12.661988,5.972368
1,2014-01-02,13.752493,8.764956
2,2014-01-03,14.963504,9.397372
3,2014-01-04,13.499846,5.309846
4,2014-01-05,12.584919,4.452709


## Dataset de Festivos y Laborales

### Carga de ficheros y unificación en un mismo dataset

In [9]:
PREFIX_FESTIVOS = "bronze/festivos/" 

paginator = s3_client.get_paginator("list_objects_v2")
paginas = paginator.paginate(Bucket=BUCKET_NAME, Prefix=PREFIX_FESTIVOS)

lista_festivos = []

for pagina in paginas:
    if "Contents" in pagina:
        for objeto in pagina["Contents"]:
            ruta_s3 = objeto["Key"]
            
            # Buscamos los archivos que terminen en festivos_ES.json
            if "festivos_ES.json" in ruta_s3:
                respuesta = s3_client.get_object(Bucket=BUCKET_NAME, Key=ruta_s3)
                contenido_json = json.loads(respuesta["Body"].read().decode("utf-8"))
                
                # Nager.Date suele devolver una lista directa de festivos
                df_año = pd.DataFrame(contenido_json)
                lista_festivos.append(df_año)

if lista_festivos:
    df_festivos = pd.concat(lista_festivos, ignore_index=True)
    print(f"Dataset de Festivos cargado. Total de días festivos registrados: {df_festivos.shape[0]}")
    display(df_festivos.head())
else:
    print("No se encontraron archivos 'festivos_ES.json' en el bucket.")

Dataset de Festivos cargado. Total de días festivos registrados: 424


,date,localName,name,countryCode,fixed,global,counties,launchYear,types
0,2014-01-01,Año Nuevo,New Year's Day,ES,False,True,None,None,[Public]
1,2014-01-06,Día de Reyes / Epifanía del Señor,Epiphany,ES,False,True,None,None,[Public]
2,2014-02-28,Día de Andalucía,Day of Andalucía,ES,False,False,[ES-AN],None,[Public]
3,2014-03-01,Dia de les Illes Balears,Day of the Balearic Islands,ES,False,False,[ES-IB],None,[Public]
4,2014-03-19,San José,Saint Joseph's Day,ES,False,False,"[ES-AR, ES-CL, ES-CM, ES-EX, ES-GA, ES-MD, ES-...",None,[Public]


### Análisis estadístico (EDA)

In [10]:
print("\nEstructura y Tipos de Datos:")
df_festivos.info()

print("\nValores Nulos por Columna:")
nulos_festivos = df_festivos.isnull().sum()
print(nulos_festivos[nulos_festivos > 0] if nulos_festivos.sum() > 0 else "¡Perfecto! Sin nulos.")

print("\nNombres de columnas disponibles:")
print(df_festivos.columns.tolist())


Estructura y Tipos de Datos:
<class 'pandas.DataFrame'>
RangeIndex: 424 entries, 0 to 423
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   date         424 non-null    str   
 1   localName    424 non-null    str   
 2   name         424 non-null    str   
 3   countryCode  424 non-null    str   
 4   fixed        424 non-null    bool  
 5   global       424 non-null    bool  
 6   counties     295 non-null    object
 7   launchYear   0 non-null      object
 8   types        424 non-null    object
dtypes: bool(2), object(3), str(4)
memory usage: 45.3+ KB

Valores Nulos por Columna:
counties      129
launchYear    424
dtype: int64

Nombres de columnas disponibles:
['date', 'localName', 'name', 'countryCode', 'fixed', 'global', 'counties', 'launchYear', 'types']


### Limpieza del Dataframe de Festivos

In [11]:
df_festivos_plata = df_festivos.copy()

# Renombramos 'date' para que actúe como llave maestra en el cruce
df_festivos_plata = df_festivos.rename(columns={'date': 'fecha_cruce', 'localName': 'nombre_festivo', 'global': 'festivo_nacional'})

# Nos quedamos solo con las columnas importantes
df_festivos_plata = df_festivos_plata[['fecha_cruce', 'nombre_festivo', 'festivo_nacional']]

df_festivos_plata.head()

,fecha_cruce,nombre_festivo,festivo_nacional
0,2014-01-01,Año Nuevo,True
1,2014-01-06,Día de Reyes / Epifanía del Señor,True
2,2014-02-28,Día de Andalucía,False
3,2014-03-01,Dia de les Illes Balears,False
4,2014-03-19,San José,False


## Generamos el dataframe Maestro uniendo los tres anteriores ya tratados convirtiéndolos así en datos de la capa oro

Los modelos de Machine Learning entienden mejor números enteros que fechas, horas y zonas horarias, así que dividimos la columna hora en una columna hora y otra minutos, descartando la zona horaria para obtener una mejor predicción.

In [12]:
# Primer cruce: Demanda + Clima. Hacemos Left Join para asegurar que no perdemos ni un solo dato de demanda
df_maestro_oro = pd.merge(df_demanda_plata, df_clima_plata, on='fecha_cruce', how='left')

# Segundo cruce: (Demanda + Clima) + Festivos
df_maestro_oro = pd.merge(df_maestro_oro, df_festivos_plata, on='fecha_cruce', how='left')


# Vemos cómo queda nuestra tabla
print(f"Filas totales de la tabla maestra: {df_maestro_oro.shape[0]}")
display(df_maestro_oro.head(300))
df_maestro_oro.info()

Filas totales de la tabla maestra: 828312


,fecha_cruce,hora,demanda_mw,temperatura_maxima,temperatura_minima,nombre_festivo,festivo_nacional
0,2014-01-01,00:00:00,24546.0,12.661988,5.972368,Año Nuevo,True
1,2014-01-01,00:10:00,24309.0,12.661988,5.972368,Año Nuevo,True
2,2014-01-01,00:20:00,24348.0,12.661988,5.972368,Año Nuevo,True
3,2014-01-01,00:30:00,24321.0,12.661988,5.972368,Año Nuevo,True
4,2014-01-01,00:40:00,24194.0,12.661988,5.972368,Año Nuevo,True
...,...,...,...,...,...,...,...
295,2014-01-03,01:10:00,24243.0,14.963504,9.397372,NaN,NaN
296,2014-01-03,01:20:00,24173.0,14.963504,9.397372,NaN,NaN
297,2014-01-03,01:30:00,23777.0,14.963504,9.397372,NaN,NaN
298,2014-01-03,01:40:00,23293.0,14.963504,9.397372,NaN,NaN


<class 'pandas.DataFrame'>
RangeIndex: 828312 entries, 0 to 828311
Data columns (total 7 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   fecha_cruce         828312 non-null  str    
 1   hora                828312 non-null  str    
 2   demanda_mw          828312 non-null  float64
 3   temperatura_maxima  826440 non-null  float64
 4   temperatura_minima  826440 non-null  float64
 5   nombre_festivo      73584 non-null   str    
 6   festivo_nacional    73584 non-null   object 
dtypes: float64(3), object(1), str(3)
memory usage: 60.2+ MB


### Transformación y arreglo del Dataframe

Como podemos comprobar, existen algunos valores nulos en las columnas de temperatura, surgidos a raíz de haber hecho el Left Join. Pasamos a eliminarlos.

In [13]:
# Arreglamos con una interpolación lineal los nulos que quedaron en las temperaturas tras el cruce. 
cols_temp = ['temperatura_maxima', 'temperatura_minima']
# Interpolamos para rellenar huecos. ffill() y bfill() por si el nulo cayó en el primer o último día
df_maestro_oro[cols_temp] = df_maestro_oro[cols_temp].interpolate().bfill().ffill() 

# Extraemos los días de la semana y los meses convertidos a un número entero para mejor interpretación del modelo de MAchine Learning
df_maestro_oro['fecha'] = pd.to_datetime(df_maestro_oro['fecha_cruce'])
df_maestro_oro['mes'] = df_maestro_oro['fecha'].dt.month
df_maestro_oro['dia_semana'] = df_maestro_oro['fecha'].dt.dayofweek

# Clasificamos los festivos en 3 categorías: 0 para días laborables, 1 para festivos estándar y 2 para festivos de gran impacto como la Navidad.
festivos_top = ['Navidad', 'Año Nuevo', 'Día de Reyes / Epifanía del Señor', 'Nochebuena', 'Nochevieja'] 

# Reemplazamos los valores nulos en festivo_nacional y nombre_festivo por un 0 (Laborable) y aseguramos que sea entero
df_maestro_oro['festivo_nacional'] = df_maestro_oro['festivo_nacional'].fillna(False)
df_maestro_oro['nombre_festivo'] = df_maestro_oro['nombre_festivo'].fillna('Laborable')

def clasificar_festivo(nombre):
    if nombre == 'Laborable': 
        return 0 
    elif nombre in festivos_top: 
        return 2 
    else: 
        return 1 

# Creamos la columna numérica con esos datos
df_maestro_oro['tipo_dia'] = df_maestro_oro['nombre_festivo'].apply(clasificar_festivo)

# Al hacer el cruce, los días que NO son festivos se quedarán con valores Nulos. Los rellenamos para que la IA entienda que son días normales.
df_maestro_oro['festivo_nacional'] = df_maestro_oro['festivo_nacional'].fillna(False)
df_maestro_oro['nombre_festivo'] = df_maestro_oro['nombre_festivo'].fillna('Laborable')

# Extraemos la hora
df_maestro_oro['hora_num'] = df_maestro_oro['hora'].astype(str).str.split(':').str[0].astype(int)

# Extraemos los minutos
df_maestro_oro['minutos_num'] = df_maestro_oro['hora'].astype(str).str.split(':').str[1].astype(int)

# Eliminamos las columnas que han sido transformadas
columnas_basura = ['fecha_cruce', 'festivo_nacional']
df_maestro_oro = df_maestro_oro.drop(columns=columnas_basura)

# Ordenamos las columnas para que sea más fácil visualizarlas
columnas_ordenadas = [
    'fecha', 
    'hora_num', 
    'minutos_num', 
    'nombre_festivo',
    'mes',
    'dia_semana',
    'tipo_dia',
    'temperatura_maxima', 
    'temperatura_minima', 
    'demanda_mw'
]

# Debido al "apagón" que sufrió la península el 28 de abril de 2025, eliminamos ese día para que el modelo no se confunda con un día atípico que no se corresponde con la realidad de la demanda
df_maestro_oro = df_maestro_oro[df_maestro_oro['fecha'].dt.strftime('%Y-%m-%d') != "2025-04-28"]

# Aplicamos el nuevo orden al dataframe
df_maestro_oro = df_maestro_oro[columnas_ordenadas]

df_maestro_oro.rename(columns={'hora_num': 'hora', 'minutos_num': 'minuto'}, inplace=True)

print(df_maestro_oro.info())
display(df_maestro_oro.head())

<class 'pandas.DataFrame'>
Index: 828024 entries, 0 to 828311
Data columns (total 10 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   fecha               828024 non-null  datetime64[us]
 1   hora                828024 non-null  int64         
 2   minuto              828024 non-null  int64         
 3   nombre_festivo      828024 non-null  str           
 4   mes                 828024 non-null  int32         
 5   dia_semana          828024 non-null  int32         
 6   tipo_dia            828024 non-null  int64         
 7   temperatura_maxima  828024 non-null  float64       
 8   temperatura_minima  828024 non-null  float64       
 9   demanda_mw          828024 non-null  float64       
dtypes: datetime64[us](1), float64(3), int32(2), int64(3), str(1)
memory usage: 71.2 MB
None


,fecha,hora,minuto,nombre_festivo,mes,dia_semana,tipo_dia,temperatura_maxima,temperatura_minima,demanda_mw
0,2014-01-01,0,0,Año Nuevo,1,2,2,12.661988,5.972368,24546.0
1,2014-01-01,0,10,Año Nuevo,1,2,2,12.661988,5.972368,24309.0
2,2014-01-01,0,20,Año Nuevo,1,2,2,12.661988,5.972368,24348.0
3,2014-01-01,0,30,Año Nuevo,1,2,2,12.661988,5.972368,24321.0
4,2014-01-01,0,40,Año Nuevo,1,2,2,12.661988,5.972368,24194.0


### Exportamos el dataframe en un fichero en formato .Parquet

In [ ]:
# Guardamos el Parquet particionado SIN ensuciar el dataframe original
df_maestro_oro.assign(
    anio=df_maestro_oro['fecha'].dt.year # Creamos la columna anio solo para el guardado
).to_parquet(
    'data/gold/dataset_demanda', 
    partition_cols=['anio', 'mes'], # Usamos el año fantasma y el mes que ya tenías
    index=False
)

print("¡Capa Oro guardada y particionada con éxito!")

¡Capa Oro guardada y particionada con éxito!
